# Introduction to Vector Databases: Installing and Running ChromaDB

**Runs on:** Google Colab (free CPU), no API key or cloud account.

### What you will learn
1. What a vector database is and why it is needed
2. Installing ChromaDB and creating a client
3. Creating a collection and adding documents
4. Semantic querying and metadata filtering
5. Using your own Hugging Face embedding model
6. Updating, deleting, and persistent storage

## 1. Why a Vector Database?

| Normal Database | Vector Database |
|---|---|
| Matches exact text or values | Matches by meaning |
| SQL queries (`WHERE name = 'dog'`) | Similarity search |
| Structured data | Text, image and audio embeddings |

**ChromaDB** is a free, open-source vector database that runs locally.

In [ ]:
!pip install -q chromadb

## 2. Create a Client
`chromadb.Client()` is an in-memory client, so data is lost when the runtime restarts.

In [ ]:
import chromadb

client = chromadb.Client()
print("ChromaDB client created. Version:", chromadb.__version__)

## 3. Create a Collection (like a table)

In [ ]:
collection = client.get_or_create_collection(name="travel_notes")
print("Collection:", collection.name, "| documents:", collection.count())

## 4. Add Documents
We pass plain text. ChromaDB automatically converts it to embeddings with its built-in default model.

In [ ]:
collection.add(
    documents=[
        "The Taj Mahal in Agra is a white marble monument built by Shah Jahan.",
        "Goa is famous for its beaches, nightlife and Portuguese heritage.",
        "Kerala backwaters offer peaceful houseboat cruises.",
        "The Himalayas are ideal for trekking and mountain adventures.",
        "Jaipur, the Pink City, is known for its palaces and forts.",
        "Chennai Marina Beach is one of the longest urban beaches in India.",
    ],
    ids=["t1", "t2", "t3", "t4", "t5", "t6"],
)
print("Documents added. Total:", collection.count())

## 5. Query by Meaning
Lower `distance` means a closer match. The query shares almost no words with the best result.

In [ ]:
def show(results):
    for doc, dist in zip(results["documents"][0], results["distances"][0]):
        print(f"distance={dist:.4f} | {doc}")
    print()

show(collection.query(query_texts=["Where can I go for a mountain hike?"], n_results=2))
show(collection.query(query_texts=["Relaxing time on the water"], n_results=2))

## 6. Metadata and Filtering

In [ ]:
collection.add(
    documents=[
        "Mysore Palace is lit up with thousands of lights during Dasara.",
        "Ooty is a hill station with tea gardens and cool weather.",
    ],
    metadatas=[{"type": "heritage"}, {"type": "hill_station"}],
    ids=["t7", "t8"],
)

show(collection.query(
    query_texts=["a cool place to relax"],
    n_results=2,
    where={"type": "hill_station"},
))

## 7. Use Your Own Hugging Face Embeddings
We plug the `all-MiniLM-L6-v2` model from the embeddings notebook into a new collection.

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from chromadb.utils import embedding_functions

hf_embedder = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

custom = client.get_or_create_collection(name="pets_hf", embedding_function=hf_embedder)
custom.add(
    documents=["I adore kittens", "Puppies are so playful", "The cricket match was postponed due to rain"],
    ids=["p1", "p2", "p3"],
)

show(custom.query(query_texts=["animal lovers"], n_results=2))

## 8. Update and Delete

In [ ]:
collection.update(ids=["t6"], documents=["Chennai Marina Beach is a long sandy beach on the Bay of Bengal."])
collection.delete(ids=["t5"])

print("Remaining documents:", collection.count())
print(collection.get(ids=["t6"])["documents"])

## 9. Persistent Storage
`PersistentClient` saves data to disk so it survives a restart.

In [ ]:
persistent_client = chromadb.PersistentClient(path="./chroma_store")
saved = persistent_client.get_or_create_collection(name="persistent_demo")

saved.upsert(documents=["This record is stored on disk."], ids=["s1"])
print("Saved to ./chroma_store | count:", saved.count())

## 10. Practice Exercises (Solved)
**1 and 2.** New collection with 5 sentences and a `category` for each, filtered by two categories.

In [ ]:
practice = client.get_or_create_collection(name="practice")
practice.add(
    documents=[
        "Messi scored twice in the football final.",
        "The new phone has a faster processor.",
        "The cake needs two cups of flour.",
        "Federer won his match in straight sets.",
        "The laptop ships with 16 GB of RAM.",
    ],
    metadatas=[{"category": "sports"}, {"category": "tech"}, {"category": "food"},
               {"category": "sports"}, {"category": "tech"}],
    ids=["x1", "x2", "x3", "x4", "x5"],
)

for cat in ["sports", "tech"]:
    print("Category:", cat)
    show(practice.query(query_texts=["a great game"], n_results=2, where={"category": cat}))

In [ ]:
# Exercise 3: swap the embedding model and compare distances
mpnet = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-mpnet-base-v2")
compare = client.get_or_create_collection(name="pets_mpnet", embedding_function=mpnet)
compare.add(documents=["I adore kittens", "Puppies are so playful", "The cricket match was postponed due to rain"],
            ids=["p1", "p2", "p3"])

print("all-MiniLM-L6-v2:"); show(custom.query(query_texts=["animal lovers"], n_results=2))
print("all-mpnet-base-v2:"); show(compare.query(query_texts=["animal lovers"], n_results=2))

## Key Takeaways

| Concept | Meaning |
|---|---|
| Vector database | Stores embeddings and searches by similarity |
| Collection | Like a table: documents, embeddings, metadata |
| `add()` | Insert documents (embeddings auto-generated) |
| `query()` | Semantic search, returns closest matches and distances |
| `where={}` | Metadata filter |
| `embedding_function` | Choose your own embedding model |
| `PersistentClient` | Saves the database to disk |